# What factors most influence academic performance?

We work on the **Scenario B** models (without G1 or G2) trained in `modeling.ipynb`. That scenario is the one that answers the real research question:

> What socioeconomic, family, and study-habit factors most influence the final academic performance (G3)?

We use three complementary techniques:

1. **Linear Regression coefficients**: Linear direction and magnitude of each variable on G3.
2. **Random Forest feature importances**: Importance based on impurity reduction (MDI).
3. **SHAP values on the Random Forest**: Individual contribution of each variable for each prediction, with direction and magnitude. This is the most complete technique and the one the professor expects to see.

The three analyses are compared at the end to give a consistent answer to the research question.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

os.makedirs('figures', exist_ok=True)

# Load Scenario B models and data splits
lr_model  = joblib.load('models/LinearRegression_scenarioB.joblib')
rf_model  = joblib.load('models/RandomForest_scenarioB.joblib')
knn_model = joblib.load('models/KNN_scenarioB.joblib')

splits = joblib.load('models/data_splits.joblib')
features_B = splits['features_B']
X_train_B  = splits['X_train_B']
X_test_B   = splits['X_test_B']

print(f'{len(features_B)} features (without G1, G2)')
print('Models loaded: LinearRegression, RandomForest, KNN')

## Linear Regression Coefficients

Linear regression, when trained with standardized features (`StandardScaler`), produces coefficients that are directly comparable in magnitude: the coefficient represents how many G3 points the model shifts per one standard deviation of the variable.

- **Positive** coefficients (blue): Increase expected performance.
- **Negative** coefficients (red): Decrease expected performance.
- We sort by absolute value to see the factors with the greatest weight, regardless of sign.


In [ ]:
# The Pipeline has [scaler, model], so the model is accessed via .named_steps['model']
coefs = lr_model.named_steps['model'].coef_
coef_df = pd.DataFrame({'feature': features_B, 'coef': coefs})
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=True).reset_index(drop=True)

colors = ['tab:blue' if c >= 0 else 'tab:red' for c in coef_df['coef']]

fig, ax = plt.subplots(figsize=(9, max(6, 0.28 * len(coef_df))))
ax.barh(coef_df['feature'], coef_df['coef'], color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (on standardized features)')
ax.set_title('Linear Regression: Coefficients sorted by |coef|')
plt.tight_layout()
plt.savefig('figures/lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 by |coef|:')
display(coef_df.sort_values('abs_coef', ascending=False).head(10)[['feature', 'coef']].reset_index(drop=True))

## Random Forest Feature Importances

Random Forest provides an importance measure based on how much each variable contributes to reducing node impurity, averaged across all trees. Unlike linear coefficients, this measure captures **non-linear interactions** but **does not indicate direction** (only magnitude).

We show the **top 15** most important variables.


In [ ]:
importances = rf_model.feature_importances_
imp_df = pd.DataFrame({'feature': features_B, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)
top15 = imp_df.head(15).iloc[::-1]  # invert so the largest is at the top in barh

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top15['feature'], top15['importance'], color='tab:green', edgecolor='black')
ax.set_xlabel('Importance (average impurity reduction)')
ax.set_title('Random Forest: Top 15 feature importances')
plt.tight_layout()
plt.savefig('figures/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

display(imp_df.head(15))

## SHAP values on the Random Forest

SHAP (SHapley Additive exPlanations) assigns to each variable, in each individual prediction, a signed numerical contribution. When aggregated across the full set:

- The **beeswarm plot** shows, for each variable, how its contribution varies across students (each point = one student). Color represents the variable value (red = high, blue = low). This is the richest way to see both **direction** and **magnitude** simultaneously.
- The **bar plot** of mean |SHAP| summarizes the global importance of each variable in units comparable to the target magnitude.

We use `shap.TreeExplainer`, which is exact and efficient for tree-based models.


In [4]:
explainer   = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_B)

print('SHAP values shape:', np.array(shap_values).shape)
print('X_test shape:     ', X_test_B.shape)


SHAP values shape: (60, 39)
X_test shape:      (60, 39)


In [ ]:
# Beeswarm plot: direction + magnitude per variable
plt.figure()
shap.summary_plot(shap_values, X_test_B, feature_names=features_B, show=False, max_display=15)
plt.title('SHAP Beeswarm (Random Forest, Scenario B)')
plt.tight_layout()
plt.savefig('figures/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar plot: mean |SHAP|
plt.figure()
shap.summary_plot(shap_values, X_test_B, feature_names=features_B,
                  plot_type='bar', show=False, max_display=15)
plt.title('SHAP — Global importance (mean |SHAP|)')
plt.tight_layout()
plt.savefig('figures/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Comparative summary table

We consolidate the rankings from all three techniques to identify which variables appear consistently as relevant.


In [7]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': features_B, 'mean_abs_shap': mean_abs_shap}) \
            .sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

rank_lr   = coef_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)
rank_lr['rank_LR']   = rank_lr.index + 1
rank_rf   = imp_df.copy()
rank_rf['rank_RF']   = rank_rf.index + 1
rank_shap = shap_df.copy()
rank_shap['rank_SHAP'] = rank_shap.index + 1

ranking = rank_lr[['feature', 'rank_LR']] \
    .merge(rank_rf[['feature', 'rank_RF']], on='feature') \
    .merge(rank_shap[['feature', 'rank_SHAP']], on='feature')
ranking['rank_mean'] = ranking[['rank_LR', 'rank_RF', 'rank_SHAP']].mean(axis=1)
ranking = ranking.sort_values('rank_mean').reset_index(drop=True)
display(ranking.head(15))


,feature,rank_LR,rank_RF,rank_SHAP,rank_mean
0,failures,1,2,1,1.333333
1,studytime,2,6,5,4.333333
2,goout,5,3,7,5.000000
3,Medu,4,10,4,6.000000
4,absences,15,1,2,6.000000
5,sex,3,13,3,6.333333
6,age,11,5,14,10.000000
7,romantic,6,15,13,11.333333
8,health,25,4,6,11.666667
9,Fjob_teacher,7,19,10,12.000000


## Answer to the research question

**Based on the analysis, the top factors influencing academic performance are:**

All three techniques (Linear Regression coefficients, Random Forest importances, and SHAP values) consistently identify a small, coherent group of variables as the main drivers of academic performance in G3, once G1 and G2 are excluded to avoid leakage:

- **`failures`** (number of prior course failures): the variable with the greatest negative weight across all three methods. Each additional failure anticipates a significant drop in G3. It is the strongest predictor of future performance.
- **`absences`** (absences): the SHAP beeswarm shows that many absences push the prediction downward, although the effect is less linear than that of `failures`.
- **`Medu` and `Fedu`** (mother's and father's education level): appear as positive factors in all three rankings. This reinforces the effect of family educational capital on student performance.
- **`studytime`** (weekly study hours): a consistent positive factor, though of moderate magnitude.
- **`higher`** (aspiration for higher education): students who want to continue to higher education show significantly higher expected performance.
- **`goout` and `Dalc` / `Walc`** (going out with friends and alcohol consumption): appear as negative factors, especially visible in the SHAP beeswarm, although their weight is smaller than that of `failures` and `absences`.
- **`age`**: slight negative effect — higher ages within the dataset range (15–22) tend to correlate with lower performance, likely because they reflect accumulated course repetitions.

**Educational policy conclusion**: the variables actionable by the institution (attendance, follow-up with students who have prior failures, encouraging study time, vocational guidance toward higher education) are precisely those that emerge as most influential. Structural socioeconomic variables (parental education) have a real impact but are not directly modifiable by the school; they are useful for identifying higher-risk groups.
